In [2]:
import torch

# CUDA GPU 연결 확인
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

if device.type == 'cuda':
    print(f'GPU Name: {torch.cuda.get_device_name(0)}')
    print(f'GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB')
    print(f'CUDA Version: {torch.version.cuda}')
else:
    print('CUDA is not available. Using CPU.')


Using device: cuda
GPU Name: NVIDIA GeForce RTX 4060 Ti
GPU Memory: 8.00 GB
CUDA Version: 11.8


In [3]:
import os
from pathlib import Path
import cv2
import numpy as np
import pandas as pd
from tqdm import tqdm

# ====== 1. 기본 경로 설정 ======
BASE_DIR = Path(r"C:\Users\THKIM\Desktop\3-2 프로젝트\인공지능 프로젝트\data")

RAW_DIR    = BASE_DIR / "raw"
FRAMES_DIR = BASE_DIR / "frames"
FINAL_DIR  = BASE_DIR / "final"

# raw 하위 폴더 생성
(RAW_DIR / "KoDF").mkdir(parents=True, exist_ok=True)
(RAW_DIR / "FFPP").mkdir(parents=True, exist_ok=True)
(RAW_DIR / "CelebDF").mkdir(parents=True, exist_ok=True)

# frames 하위 폴더 생성
for dataset in ["KoDF", "FFPP", "CelebDF"]:
    for split in ["train", "val", "test"]:
        for label in ["real", "fake"]:
            (FRAMES_DIR / dataset / split / label).mkdir(parents=True, exist_ok=True)

# final 폴더 생성
for split in ["train", "val", "test"]:
    for label in ["real", "fake"]:
        (FINAL_DIR / split / label).mkdir(parents=True, exist_ok=True)

print("✅ 폴더 구조 생성 완료!")
print(f"RAW_DIR: {RAW_DIR}")
print(f"FRAMES_DIR: {FRAMES_DIR}")
print(f"FINAL_DIR: {FINAL_DIR}")


✅ 폴더 구조 생성 완료!
RAW_DIR: C:\Users\THKIM\Desktop\3-2 프로젝트\인공지능 프로젝트\data\raw
FRAMES_DIR: C:\Users\THKIM\Desktop\3-2 프로젝트\인공지능 프로젝트\data\frames
FINAL_DIR: C:\Users\THKIM\Desktop\3-2 프로젝트\인공지능 프로젝트\data\final


In [4]:
# ====== 1B. 샘플링 전역 설정 (가이드라인 기반 + 동적 보정) ======
import math

# 재현성 시드
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

# 공통 프레임 추출 파라미터
FRAME_SIZE = 224
JPEG_QUALITY = 80

# KoDF 목표 (가이드라인)
KODF_TARGET = {
    'train': {'real': 600, 'fake': 600},
    'val':   {'real': 200, 'fake': 200}
}
KODF_FRAMES_PER_VIDEO = 20

# FFPP 목표(가이드라인에서 고정 계획)
FFPP_TARGET = {
    'train': {'real': 400, 'fake': 400},
    'val':   {'real': 50,  'fake': 50},
    'test':  {'real': 50,  'fake': 50}
}
FFPP_FRAMES_PER_VIDEO = 20

# CelebDF 목표(Test 전용)
CELEBDF_TARGET = {
    'test': {'real': 300, 'fake': 600}
}
CELEBDF_FRAMES_PER_VIDEO = 10

# 평균 프레임 크기 추정 (낙관/보수)
EST_AVG_FRAME_BYTES_LOW  = 30_000   # 초기 낙관 추정
EST_AVG_FRAME_BYTES_HIGH = 150_000  # 초기 보수 추정
# 실제 측정 기반 동적 값 (없으면 None)
ACTUAL_AVG_FRAME_BYTES = None

DATASET_SIZE_BUDGET_GB = 20
DATASET_SIZE_BUDGET_BYTES = DATASET_SIZE_BUDGET_GB * (1024**3)

print("✅ 전역 샘플링 설정 로드 완료")
print("KoDF 목표 비디오 수:", KODF_TARGET)
print("FFPP 목표 비디오 수:", FFPP_TARGET)
print("CelebDF 목표 비디오 수:", CELEBDF_TARGET)
print(f"평균 프레임 크기(초기 낙관): {EST_AVG_FRAME_BYTES_LOW/1024:.1f} KB")
print(f"평균 프레임 크기(초기 보수): {EST_AVG_FRAME_BYTES_HIGH/1024:.1f} KB")
print(f"총 용량 예산: {DATASET_SIZE_BUDGET_GB} GB")
if ACTUAL_AVG_FRAME_BYTES:
    print(f"실측 평균 프레임 크기: {ACTUAL_AVG_FRAME_BYTES/1024:.1f} KB")

✅ 전역 샘플링 설정 로드 완료
KoDF 목표 비디오 수: {'train': {'real': 600, 'fake': 600}, 'val': {'real': 200, 'fake': 200}}
FFPP 목표 비디오 수: {'train': {'real': 400, 'fake': 400}, 'val': {'real': 50, 'fake': 50}, 'test': {'real': 50, 'fake': 50}}
CelebDF 목표 비디오 수: {'test': {'real': 300, 'fake': 600}}
평균 프레임 크기(초기 낙관): 29.3 KB
평균 프레임 크기(초기 보수): 146.5 KB
총 용량 예산: 20 GB


In [102]:
# ====== 2. 공통 프레임 추출 함수 (디버그 + Pillow fallback 개선) ======
from PIL import Image

def save_image_with_fallback(frame_bgr: np.ndarray, out_path: Path, jpeg_quality: int = 80, debug: bool=False):
    """우선 cv2.imwrite 시도, 실패 시 Pillow(RGB)로 재시도.
    Returns: bool(성공 여부), used_pillow(bool)
    """
    # 1차: OpenCV
    try:
        ok = cv2.imwrite(str(out_path), frame_bgr, [cv2.IMWRITE_JPEG_QUALITY, jpeg_quality])
        if ok:
            return True, False
    except Exception as e:
        if debug:
            print(f"[DEBUG] cv2.imwrite 예외: {e}")
    # 2차: Pillow fallback
    try:
        img_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
        pil_img = Image.fromarray(img_rgb)
        pil_img.save(out_path, format='JPEG', quality=jpeg_quality)
        return True, True
    except Exception as e:
        if debug:
            print(f"[DEBUG] Pillow 저장 실패: {e}")
        return False, True

def extract_frames_from_video(
    video_path: Path,
    out_dir: Path,
    video_id: str,
    num_samples: int = 8,
    size: int = 224,
    jpeg_quality: int = 80,
    debug: bool = False,
):
    """
    비디오에서 균등 간격으로 프레임 샘플링

    Args:
        video_path: 비디오 파일 경로
        out_dir: 출력 디렉토리
        video_id: 비디오 고유 ID (파일명에 사용)
        num_samples: 추출할 프레임 수
        size: 리사이즈 크기 (정사각형)
        jpeg_quality: JPEG 압축 품질 (0-100)
        debug: True면 첫/실패 프레임 경로 및 오류 상세 로그

    Returns:
        저장된 프레임 수
    """
    out_dir.mkdir(parents=True, exist_ok=True)

    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        if debug:
            print(f"[DEBUG] Cannot open video: {video_path}")
        return 0

    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if frame_count <= 0:
        cap.release()
        if debug:
            print(f"[DEBUG] Invalid frame_count (<=0): {video_path}")
        return 0

    num_samples_eff = min(num_samples, frame_count)
    indices = np.linspace(0, frame_count - 1, num_samples_eff, dtype=int)

    saved = 0
    first_written_path = None
    write_failures = 0
    read_failures = 0
    pillow_used_any = False

    for idx in indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(idx))
        ret, frame = cap.read()
        if not ret or frame is None:
            read_failures += 1
            if debug and read_failures <= 3:
                print(f"[DEBUG] Read failure idx={idx} video={video_path}")
            continue

        try:
            frame = cv2.resize(frame, (size, size))
        except Exception as e:
            if debug:
                print(f"[DEBUG] resize 실패 idx={idx}: {e}")
            continue

        fname = f"{video_id}_f{idx:05d}.jpg"
        out_path = out_dir / fname

        ok, pillow_used = save_image_with_fallback(frame, out_path, jpeg_quality, debug=debug)
        if not ok:
            write_failures += 1
            if debug and write_failures <= 5:
                print(f"[DEBUG] 최종 저장 실패 path={out_path}")
            continue
        if pillow_used:
            pillow_used_any = True
        if first_written_path is None:
            first_written_path = out_path
        saved += 1

    cap.release()

    if debug:
        print(f"[DEBUG] Saved {saved}/{num_samples_eff} frames for video_id={video_id}")
        if first_written_path:
            try:
                sz = Path(first_written_path).stat().st_size
                print(f"[DEBUG] First frame path={first_written_path} size={sz} bytes")
            except Exception as e:
                print(f"[DEBUG] Stat failure {first_written_path}: {e}")
        if read_failures:
            print(f"[DEBUG] Read failures: {read_failures}")
        if write_failures:
            print(f"[DEBUG] Write failures: {write_failures}")
        if pillow_used_any:
            print(f"[DEBUG] Pillow fallback 사용됨")

    return saved

print("✅ 프레임 추출 함수 재정의 완료! (Pillow fallback 포함)")

✅ 프레임 추출 함수 재정의 완료! (Pillow fallback 포함)


In [103]:
# ====== 3. KoDF (AIHub 딥페이크 변조 영상) 샘플링 ======
def sample_kodf_train_val():
    """
    KoDF / AIHub 데이터셋 샘플링 (가이드라인 기반 목표치 적용)
    - 폴더 구조: UUID 폴더 하위 여러 mp4
    - 목표 비디오 수(KODF_TARGET) 이상 존재 시 랜덤 샘플링
    - 부족 시 전체 사용 + 경고 출력
    - 비디오당 프레임 수: KODF_FRAMES_PER_VIDEO
    - 용량 추정: 낙관(LOW) vs 보수(HIGH) 모두 출력
    """
    print("\n" + "="*60)
    print("KoDF 샘플링 시작 (가이드라인 적용)")
    print("="*60)

    # 비디오 루트 경로 정의
    video_root_train_fake = BASE_DIR / "딥페이크 변조 영상" / "1.Training" / "[원천][변조]train_dfl1_data" / "dfl1"
    video_root_train_real = BASE_DIR / "딥페이크 변조 영상" / "1.Training" / "[원천][원본]원본1" / "원본1"
    video_root_val_fake   = BASE_DIR / "딥페이크 변조 영상" / "2.Validation" / "[원천][변조]vaild_dfl_data" / "dfl"
    video_root_val_real   = BASE_DIR / "딥페이크 변조 영상" / "2.Validation" / "[원천][원본]원본1" / "원본1"

    def collect_videos(root: Path):
        collected = []
        if root.exists():
            for uuid_folder in root.glob("*"):
                if uuid_folder.is_dir():
                    for vf in uuid_folder.glob("*.mp4"):
                        collected.append((uuid_folder.name, vf))
        return collected

    # 수집
    train_real_all = collect_videos(video_root_train_real)
    train_fake_all = collect_videos(video_root_train_fake)
    val_real_all   = collect_videos(video_root_val_real)
    val_fake_all   = collect_videos(video_root_val_fake)

    print(f"발견된 Train Real 비디오: {len(train_real_all)}")
    print(f"발견된 Train Fake 비디오: {len(train_fake_all)}")
    print(f"발견된 Val   Real 비디오: {len(val_real_all)}")
    print(f"발견된 Val   Fake 비디오: {len(val_fake_all)}")

    # 목표 설정
    tgt_tr_real = KODF_TARGET['train']['real']
    tgt_tr_fake = KODF_TARGET['train']['fake']
    tgt_val_real = KODF_TARGET['val']['real']
    tgt_val_fake = KODF_TARGET['val']['fake']

    def select_subset(all_list, target, label_desc):
        if len(all_list) >= target:
            idx = np.random.permutation(len(all_list))[:target]
            return [all_list[i] for i in idx], False
        else:
            print(f"[WARN] {label_desc} 목표 {target}개 대비 실제 {len(all_list)}개 → 전체 사용")
            return all_list, True

    train_real_sel, _ = select_subset(train_real_all, tgt_tr_real, 'Train Real')
    train_fake_sel, _ = select_subset(train_fake_all, tgt_tr_fake, 'Train Fake')
    val_real_sel,   _ = select_subset(val_real_all,  tgt_val_real, 'Val Real')
    val_fake_sel,   _ = select_subset(val_fake_all,  tgt_val_fake, 'Val Fake')

    print("\n[선택 결과]")
    print(f"Train Real: {len(train_real_sel)} / 목표 {tgt_tr_real}")
    print(f"Train Fake: {len(train_fake_sel)} / 목표 {tgt_tr_fake}")
    print(f"Val   Real: {len(val_real_sel)} / 목표 {tgt_val_real}")
    print(f"Val   Fake: {len(val_fake_sel)} / 목표 {tgt_val_fake}")

    # 예상 프레임 및 용량 계산(낙관/보수)
    total_videos = (len(train_real_sel) + len(train_fake_sel) + len(val_real_sel) + len(val_fake_sel))
    expected_frames = total_videos * KODF_FRAMES_PER_VIDEO
    bytes_low  = expected_frames * EST_AVG_FRAME_BYTES_LOW
    bytes_high = expected_frames * EST_AVG_FRAME_BYTES_HIGH
    gb_low  = bytes_low  / (1024**3)
    gb_high = bytes_high / (1024**3)
    print(f"\n[KoDF 예상 프레임] {expected_frames:,} 프레임")
    print(f"[KoDF 예상 용량 낙관] ≈ {gb_low:.2f} GB @ {EST_AVG_FRAME_BYTES_LOW/1024:.0f}KB")
    print(f"[KoDF 예상 용량 보수] ≈ {gb_high:.2f} GB @ {EST_AVG_FRAME_BYTES_HIGH/1024:.0f}KB")

    def extract_batch(video_tuples, split, label):
        out_dir = FRAMES_DIR / "KoDF" / split / label
        for uuid, vpath in tqdm(video_tuples, desc=f"KoDF {split} {label}"):
            vid_id = vpath.stem
            extract_frames_from_video(
                video_path=vpath,
                out_dir=out_dir,
                video_id=f"kodf_{split}_{label}_{uuid}_{vid_id}",
                num_samples=KODF_FRAMES_PER_VIDEO,
                size=FRAME_SIZE,
                jpeg_quality=JPEG_QUALITY,
            )

    print("\n[1/4] Train Real 프레임 추출...")
    extract_batch(train_real_sel, 'train', 'real')
    print("\n[2/4] Train Fake 프레임 추출...")
    extract_batch(train_fake_sel, 'train', 'fake')
    print("\n[3/4] Val Real 프레임 추출...")
    extract_batch(val_real_sel, 'val', 'real')
    print("\n[4/4] Val Fake 프레임 추출...")
    extract_batch(val_fake_sel, 'val', 'fake')

    print("\n✅ KoDF 샘플링 완료 (가이드라인 적용)")

print("✅ KoDF 샘플링 함수 정의 완료! (가이드라인 버전)")

✅ KoDF 샘플링 함수 정의 완료! (가이드라인 버전)


In [6]:
# ====== 4. FaceForensics++ 샘플링 ======
def sample_ffpp():
    """
    FaceForensics++ 데이터셋 샘플링
    - CSV 메타데이터로 Label 정보 확인
    - 여러 조작 기법 혼합: Deepfakes, Face2Face, FaceSwap, NeuralTextures 등
    """
    print("\n" + "="*60)
    print("FaceForensics++ 샘플링 시작")
    print("="*60)
    
    # 메타데이터 경로
    meta_path = BASE_DIR / "FF++ datasets" / "FaceForensics++_C23" / "csv" / "FF++_Metadata.csv"
    video_root = BASE_DIR / "FF++ datasets" / "FaceForensics++_C23"
    
    # 메타데이터 로드
    meta_df = pd.read_csv(meta_path)
    print(f"Total videos: {len(meta_df)}")
    print(f"Label distribution:\n{meta_df['Label'].value_counts()}")
    
    # Real / Fake 분리
    real_meta = meta_df[meta_df['Label'] == 'REAL'].copy()
    fake_meta = meta_df[meta_df['Label'] == 'FAKE'].copy()
    
    # 샘플링 (Train 400 + Val 50 + Test 50 = 총 500개씩)
    N_TRAIN_REAL = min(400, len(real_meta))
    N_TRAIN_FAKE = min(400, len(fake_meta))
    N_VAL_REAL = min(50, len(real_meta) - N_TRAIN_REAL)
    N_VAL_FAKE = min(50, len(fake_meta) - N_TRAIN_FAKE)
    N_TEST_REAL = min(50, len(real_meta) - N_TRAIN_REAL - N_VAL_REAL)
    N_TEST_FAKE = min(50, len(fake_meta) - N_TRAIN_FAKE - N_VAL_FAKE)
    
    # 랜덤 샘플링으로 분할
    real_shuffled = real_meta.sample(frac=1, random_state=42).reset_index(drop=True)
    fake_shuffled = fake_meta.sample(frac=1, random_state=42).reset_index(drop=True)
    
    train_real_sample = real_shuffled[:N_TRAIN_REAL]
    val_real_sample = real_shuffled[N_TRAIN_REAL:N_TRAIN_REAL+N_VAL_REAL]
    test_real_sample = real_shuffled[N_TRAIN_REAL+N_VAL_REAL:N_TRAIN_REAL+N_VAL_REAL+N_TEST_REAL]
    
    train_fake_sample = fake_shuffled[:N_TRAIN_FAKE]
    val_fake_sample = fake_shuffled[N_TRAIN_FAKE:N_TRAIN_FAKE+N_VAL_FAKE]
    test_fake_sample = fake_shuffled[N_TRAIN_FAKE+N_VAL_FAKE:N_TRAIN_FAKE+N_VAL_FAKE+N_TEST_FAKE]
    
    print(f"\nSampled train: real {len(train_real_sample)}, fake {len(train_fake_sample)}")
    print(f"Sampled val: real {len(val_real_sample)}, fake {len(val_fake_sample)}")
    print(f"Sampled test: real {len(test_real_sample)}, fake {len(test_fake_sample)}")
    
    # 1) Train Real
    print("\n[1/6] Train Real 샘플링...")
    for idx, row in tqdm(train_real_sample.iterrows(), total=len(train_real_sample), desc="FFPP train real"):
        file_path = row['File Path']
        video_path = video_root / file_path
        
        if not video_path.exists():
            continue
        
        video_id = f"ffpp_tr_real_{Path(file_path).stem}"
        
        out_dir = FRAMES_DIR / "FFPP" / "train" / "real"
        extract_frames_from_video(
            video_path=video_path,
            out_dir=out_dir,
            video_id=video_id,
            num_samples=20,
        )
    
    # 2) Train Fake
    print("\n[2/6] Train Fake 샘플링...")
    for idx, row in tqdm(train_fake_sample.iterrows(), total=len(train_fake_sample), desc="FFPP train fake"):
        file_path = row['File Path']
        video_path = video_root / file_path
        
        if not video_path.exists():
            continue
        
        video_id = f"ffpp_tr_fake_{Path(file_path).stem}"
        
        out_dir = FRAMES_DIR / "FFPP" / "train" / "fake"
        extract_frames_from_video(
            video_path=video_path,
            out_dir=out_dir,
            video_id=video_id,
            num_samples=20,
        )
    
    # 3) Val Real
    print("\n[3/6] Val Real 샘플링...")
    for idx, row in tqdm(val_real_sample.iterrows(), total=len(val_real_sample), desc="FFPP val real"):
        file_path = row['File Path']
        video_path = video_root / file_path
        
        if not video_path.exists():
            continue
        
        video_id = f"ffpp_val_real_{Path(file_path).stem}"
        
        out_dir = FRAMES_DIR / "FFPP" / "val" / "real"
        extract_frames_from_video(
            video_path=video_path,
            out_dir=out_dir,
            video_id=video_id,
            num_samples=20,
        )
    
    # 4) Val Fake
    print("\n[4/6] Val Fake 샘플링...")
    for idx, row in tqdm(val_fake_sample.iterrows(), total=len(val_fake_sample), desc="FFPP val fake"):
        file_path = row['File Path']
        video_path = video_root / file_path
        
        if not video_path.exists():
            continue
        
        video_id = f"ffpp_val_fake_{Path(file_path).stem}"
        
        out_dir = FRAMES_DIR / "FFPP" / "val" / "fake"
        extract_frames_from_video(
            video_path=video_path,
            out_dir=out_dir,
            video_id=video_id,
            num_samples=20,
        )
    
    # 5) Test Real
    print("\n[5/6] Test Real 샘플링...")
    for idx, row in tqdm(test_real_sample.iterrows(), total=len(test_real_sample), desc="FFPP test real"):
        file_path = row['File Path']
        video_path = video_root / file_path
        
        if not video_path.exists():
            continue
        
        video_id = f"ffpp_test_real_{Path(file_path).stem}"
        
        out_dir = FRAMES_DIR / "FFPP" / "test" / "real"
        extract_frames_from_video(
            video_path=video_path,
            out_dir=out_dir,
            video_id=video_id,
            num_samples=20,
        )
    
    # 6) Test Fake
    print("\n[6/6] Test Fake 샘플링...")
    for idx, row in tqdm(test_fake_sample.iterrows(), total=len(test_fake_sample), desc="FFPP test fake"):
        file_path = row['File Path']
        video_path = video_root / file_path
        
        if not video_path.exists():
            continue
        
        video_id = f"ffpp_test_fake_{Path(file_path).stem}"
        
        out_dir = FRAMES_DIR / "FFPP" / "test" / "fake"
        extract_frames_from_video(
            video_path=video_path,
            out_dir=out_dir,
            video_id=video_id,
            num_samples=20,
        )
    
    print("\n✅ FaceForensics++ 샘플링 완료!")

print("✅ FFPP 샘플링 함수 정의 완료!")

✅ FFPP 샘플링 함수 정의 완료!


In [105]:
# ====== 5. Celeb-DF 샘플링 ======
def sample_celebdf():
    """
    Celeb-DF 데이터셋 샘플링
    - List_of_testing_videos.txt에서 라벨과 경로 정보 파싱
    - 첫 번째 컬럼이 라벨 (1=real, 0=fake)
    """
    print("\n" + "="*60)
    print("Celeb-DF 샘플링 시작")
    print("="*60)
    
    # 리스트 파일 경로
    list_path = BASE_DIR / "CelebDF datasets" / "List_of_testing_videos.txt"
    video_root = BASE_DIR / "CelebDF datasets"
    
    # 리스트 파일 파싱
    real_videos = []
    fake_videos = []
    
    with open(list_path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            
            parts = line.split()
            if len(parts) < 2:
                continue
            
            label = int(parts[0])  # 1=real, 0=fake
            video_path = parts[1]
            
            full_path = video_root / video_path
            
            if label == 1:
                real_videos.append(full_path)
            else:
                fake_videos.append(full_path)
    
    print(f"Total real videos: {len(real_videos)}")
    print(f"Total fake videos: {len(fake_videos)}")
    
    # 샘플링 (Test용 - real 300, fake 600)
    N_TEST_REAL = min(300, len(real_videos))
    N_TEST_FAKE = min(600, len(fake_videos))
    
    np.random.seed(42)
    real_sample = np.random.choice(real_videos, N_TEST_REAL, replace=False)
    fake_sample = np.random.choice(fake_videos, N_TEST_FAKE, replace=False)
    
    print(f"\nSampled test real: {len(real_sample)}")
    print(f"Sampled test fake: {len(fake_sample)}")
    
    # 1) Test Real
    print("\n[1/2] Test Real 샘플링...")
    for i, video_path in enumerate(tqdm(real_sample, desc="CelebDF test real")):
        if not video_path.exists():
            continue
        
        video_id = f"celebdf_test_real_{i:04d}"
        
        out_dir = FRAMES_DIR / "CelebDF" / "test" / "real"
        extract_frames_from_video(
            video_path=video_path,
            out_dir=out_dir,
            video_id=video_id,
            num_samples=10,
        )
    
    # 2) Test Fake
    print("\n[2/2] Test Fake 샘플링...")
    for i, video_path in enumerate(tqdm(fake_sample, desc="CelebDF test fake")):
        if not video_path.exists():
            continue
        
        video_id = f"celebdf_test_fake_{i:04d}"
        
        out_dir = FRAMES_DIR / "CelebDF" / "test" / "fake"
        extract_frames_from_video(
            video_path=video_path,
            out_dir=out_dir,
            video_id=video_id,
            num_samples=10,
        )
    
    print("\n✅ Celeb-DF 샘플링 완료!")

print("✅ CelebDF 샘플링 함수 정의 완료!")


✅ CelebDF 샘플링 함수 정의 완료!


In [7]:
# ====== 6. 메인 실행부 ======
# 실행 순서:
# 1. 첫 번째 셀: GPU 연결 확인
# 2. 두 번째 셀: 폴더 구조 생성
# 1B. 전역 설정 셀
# 3~5. 각 샘플링 함수 정의
# 6. 메인 실행 함수 정의
# 7. run_all_sampling() 호출로 실제 샘플링 수행

# 예상 용량/프레임 사전 계산 헬퍼

def print_expected_plan():
    # KoDF
    kodf_train_frames = (KODF_TARGET['train']['real'] + KODF_TARGET['train']['fake']) * KODF_FRAMES_PER_VIDEO
    kodf_val_frames   = (KODF_TARGET['val']['real']   + KODF_TARGET['val']['fake'])   * KODF_FRAMES_PER_VIDEO
    # FFPP
    ffpp_train_frames = (FFPP_TARGET['train']['real'] + FFPP_TARGET['train']['fake']) * FFPP_FRAMES_PER_VIDEO
    ffpp_val_frames   = (FFPP_TARGET['val']['real']   + FFPP_TARGET['val']['fake'])   * FFPP_FRAMES_PER_VIDEO
    ffpp_test_frames  = (FFPP_TARGET['test']['real']  + FFPP_TARGET['test']['fake'])  * FFPP_FRAMES_PER_VIDEO
    # CelebDF
    celebdf_test_frames = (CELEBDF_TARGET['test']['real'] + CELEBDF_TARGET['test']['fake']) * CELEBDF_FRAMES_PER_VIDEO

    train_total = kodf_train_frames + ffpp_train_frames
    val_total   = kodf_val_frames   + ffpp_val_frames
    test_total  = ffpp_test_frames  + celebdf_test_frames
    grand_total = train_total + val_total + test_total

    # 동적 평균 (실측 있으면 반영, 없으면 낙관/보수 둘 다)
    if ACTUAL_AVG_FRAME_BYTES:
        bytes_actual = grand_total * ACTUAL_AVG_FRAME_BYTES
        gb_actual = bytes_actual / (1024**3)
    else:
        gb_actual = None

    bytes_low  = grand_total * EST_AVG_FRAME_BYTES_LOW
    bytes_high = grand_total * EST_AVG_FRAME_BYTES_HIGH
    gb_low  = bytes_low  / (1024**3)
    gb_high = bytes_high / (1024**3)

    print("\n" + "="*60)
    print("[예상 프레임/용량 계획]")
    print("="*60)
    print(f"Train: {train_total:,}  (KoDF {kodf_train_frames:,} + FFPP {ffpp_train_frames:,})")
    print(f"Val:   {val_total:,}  (KoDF {kodf_val_frames:,} + FFPP {ffpp_val_frames:,})")
    print(f"Test:  {test_total:,}  (FFPP {ffpp_test_frames:,} + CelebDF {celebdf_test_frames:,})")
    print(f"Total Frames: {grand_total:,}")
    print(f"낙관 용량 ≈ {gb_low:.2f} GB  (avg {EST_AVG_FRAME_BYTES_LOW/1024:.0f}KB)")
    print(f"보수 용량 ≈ {gb_high:.2f} GB  (avg {EST_AVG_FRAME_BYTES_HIGH/1024:.0f}KB)")
    if gb_actual is not None:
        print(f"실측 평균 기반 추정 ≈ {gb_actual:.2f} GB  (avg {ACTUAL_AVG_FRAME_BYTES/1024:.1f}KB)")
    print(f"예산 20GB 대비 낙관 여유: {(20 - gb_low):.2f} GB / 보수 여유: {(20 - gb_high):.2f} GB")
    if gb_actual is not None:
        print(f"실측 기반 여유: {(20 - gb_actual):.2f} GB")


def run_all_sampling():
    """전체 샘플링 실행"""
    print_expected_plan()

    print("\n" + "="*60)
    print("전체 데이터셋 샘플링 시작")
    print("="*60)
    print(f"작업 시작 시간: {pd.Timestamp.now()}")

    import time
    start_time = time.time()

    # 1. KoDF 샘플링
    try:
        sample_kodf_train_val()
    except Exception as e:
        print(f"[ERROR] KoDF 샘플링 실패: {e}")

    # 2. FFPP 샘플링
    try:
        sample_ffpp()
    except Exception as e:
        print(f"[ERROR] FFPP 샘플링 실패: {e}")

    # 3. CelebDF 샘플링
    try:
        sample_celebdf()
    except Exception as e:
        print(f"[ERROR] CelebDF 샘플링 실패: {e}")

    # 완료 정보
    elapsed_time = time.time() - start_time
    print("\n" + "="*60)
    print("✅ 전체 샘플링 완료!")
    print(f"작업 종료 시간: {pd.Timestamp.now()}")
    print(f"총 소요 시간: {elapsed_time/60:.2f}분")
    print("="*60)

    # 프레임 수 카운트
    print("\n[실제 샘플링 결과 통계]")
    actual_total = 0
    for dataset in ["KoDF", "FFPP", "CelebDF"]:
        for split in ["train", "val", "test"]:
            for label in ["real", "fake"]:
                frame_dir = FRAMES_DIR / dataset / split / label
                if frame_dir.exists():
                    count = len(list(frame_dir.glob("*.jpg")))
                    if count > 0:
                        print(f"{dataset}/{split}/{label}: {count} frames")
                        actual_total += count
    print(f"\n총 프레임 수(실제): {actual_total:,}")

    # 메타데이터 저장
    print("\n" + "="*60)
    print("메타데이터 저장 중...")
    try:
        save_sampling_metadata()
    except Exception as e:
        print(f"[ERROR] 메타데이터 저장 실패: {e}")

    print("\n" + "="*60)
    print("🎉 모든 작업 완료!")
    print("="*60)

print("✅ 메인 실행 함수 정의 완료!")
print("\n💡 샘플링을 시작하려면 다음을 실행하세요:")
print("   run_all_sampling()")

✅ 메인 실행 함수 정의 완료!

💡 샘플링을 시작하려면 다음을 실행하세요:
   run_all_sampling()


In [8]:
# ====== 7. 샘플링 결과 저장 함수 ======
def save_sampling_metadata():
    """
    샘플링된 프레임 정보를 CSV 파일로 저장
    - 각 프레임의 경로, 라벨, 데이터셋, split 정보 저장
    - 나중에 학습 시 쉽게 로드할 수 있도록 구성
    """
    print("\n" + "="*60)
    print("샘플링 메타데이터 저장 시작")
    print("="*60)

    all_data = []
    total_scan = 0

    # 모든 프레임 파일 수집
    for dataset in ["KoDF", "FFPP", "CelebDF"]:
        for split in ["train", "val", "test"]:
            for label in ["real", "fake"]:
                frame_dir = FRAMES_DIR / dataset / split / label
                if not frame_dir.exists():
                    continue
                files = list(frame_dir.glob("*.jpg"))
                total_scan += len(files)
                for frame_path in files:
                    all_data.append({
                        'frame_path': str(frame_path.relative_to(FRAMES_DIR)),
                        'absolute_path': str(frame_path),
                        'filename': frame_path.name,
                        'dataset': dataset,
                        'split': split,
                        'label': label,
                        'label_numeric': 1 if label == 'fake' else 0,
                    })

    print(f"[DEBUG] 총 스캔된 jpg 파일 수: {total_scan}")

    df = pd.DataFrame(all_data)

    if len(df) == 0:
        # 추가 진단: 존재하는 상위 폴더, 최초 5개 파일 트라이
        print("⚠️ 샘플링된 프레임이 없습니다! 추가 진단 정보:")
        for dataset in ["KoDF", "FFPP", "CelebDF"]:
            top = FRAMES_DIR / dataset
            if top.exists():
                sample_list = list(top.rglob("*.jpg"))[:5]
                print(f"  {dataset} jpg 샘플 5개: {[str(p) for p in sample_list]}")
            else:
                print(f"  {dataset} 폴더 없음: {top}")
        return

    print(f"\n총 프레임 수: {len(df):,}")
    print("\n[데이터셋별 통계]")
    try:
        print(df.groupby(['dataset', 'split', 'label']).size().unstack(fill_value=0))
    except Exception as e:
        print(f"[WARN] 그룹화 실패: {e}")

    print("\n[Split별 통계]")
    try:
        print(df.groupby(['split', 'label']).size().unstack(fill_value=0))
    except Exception as e:
        print(f"[WARN] 그룹화 실패: {e}")

    metadata_path = FRAMES_DIR / "sampling_metadata.csv"
    df.to_csv(metadata_path, index=False, encoding='utf-8-sig')
    print(f"\n✅ 메타데이터 저장 완료: {metadata_path}")

    for split in ['train', 'val', 'test']:
        split_df = df[df['split'] == split]
        if len(split_df) > 0:
            split_path = FRAMES_DIR / f"sampling_metadata_{split}.csv"
            split_df.to_csv(split_path, index=False, encoding='utf-8-sig')
            print(f"  - {split} 메타데이터: {split_path} ({len(split_df):,} frames)")

    return df

print("✅ 메타데이터 저장 함수 정의 완료! (디버그 강화)")

✅ 메타데이터 저장 함수 정의 완료! (디버그 강화)


In [9]:
# ====== (재배치) 메타데이터 활용/검증/실측 평균 보정 헬퍼 ======
from statistics import mean

def load_sampling_metadata(split=None):
    """저장된 샘플링 메타데이터 로드 (split 지정 가능)"""
    if split:
        metadata_path = FRAMES_DIR / f"sampling_metadata_{split}.csv"
        if not metadata_path.exists():
            print(f"⚠️ {split} 메타데이터 파일이 없습니다: {metadata_path}")
            return None
    else:
        metadata_path = FRAMES_DIR / "sampling_metadata.csv"
        if not metadata_path.exists():
            print(f"⚠️ 메타데이터 파일이 없습니다: {metadata_path}")
            return None
    df = pd.read_csv(metadata_path)
    print(f"✅ 메타데이터 로드 완료: {len(df):,} frames")
    if split:
        print(f"Split: {split}")
    print("\n[라벨별 분포]")
    print(df['label'].value_counts())
    if 'dataset' in df.columns:
        print("\n[데이터셋별 분포]")
        print(df['dataset'].value_counts())
    return df

def verify_frames(max_report_missing=10):
    """메타데이터 내 모든 프레임 파일 존재 여부 확인"""
    print("\n" + "="*60)
    print("프레임 파일 존재 여부 확인")
    print("="*60)
    metadata_path = FRAMES_DIR / "sampling_metadata.csv"
    if not metadata_path.exists():
        print("⚠️ 먼저 save_sampling_metadata()를 실행하세요!")
        return False
    df = pd.read_csv(metadata_path)
    missing_count = 0
    total_count = len(df)
    print(f"총 {total_count:,}개 프레임 확인 중...")
    for idx, row in tqdm(df.iterrows(), total=total_count, desc="Verifying frames"):
        frame_path = Path(row['absolute_path'])
        if not frame_path.exists():
            missing_count += 1
            if missing_count <= max_report_missing:
                print(f"⚠️ 누락: {frame_path}")
    if missing_count == 0:
        print(f"\n✅ 모든 프레임 파일 확인 완료! ({total_count:,}개)")
    else:
        print(f"\n⚠️ {missing_count:,}개 파일 누락 ({total_count:,}개 중)")
    return missing_count == 0

def calibrate_actual_frame_size(sample_dirs=None, max_files=500):
    """debug 또는 일부 split 디렉토리에서 실제 JPG 크기 평균 추정 후 ACTUAL_AVG_FRAME_BYTES 갱신"""
    global ACTUAL_AVG_FRAME_BYTES
    if sample_dirs is None:
        sample_dirs = [FRAMES_DIR / '_debug_real', FRAMES_DIR / '_debug_fake']
    sizes = []
    for d in sample_dirs:
        if not d.exists():
            continue
        files = list(d.glob('*.jpg'))[:max_files]
        for f in files:
            try:
                sizes.append(f.stat().st_size)
            except Exception:
                pass
    if not sizes:
        print("⚠️ 측정할 파일이 없습니다. 먼저 quick_debug_sampling() 실행하세요.")
        return None
    avg_bytes = mean(sizes)
    ACTUAL_AVG_FRAME_BYTES = int(avg_bytes)
    print(f"✅ 실측 평균 프레임 크기 갱신: {ACTUAL_AVG_FRAME_BYTES} bytes (~{ACTUAL_AVG_FRAME_BYTES/1024:.1f} KB)")
    print("   샘플 파일 수:", len(sizes))
    print("   최소/최대:", min(sizes), max(sizes))
    print("   print_expected_plan() 재호출 시 실측 기반 용량 표시")
    return ACTUAL_AVG_FRAME_BYTES

print("✅ 헬퍼 재배치 완료: load_sampling_metadata, verify_frames, calibrate_actual_frame_size 사용 가능")

✅ 헬퍼 재배치 완료: load_sampling_metadata, verify_frames, calibrate_actual_frame_size 사용 가능


In [10]:
# ====== Quick Debug Sampling (정리 버전) ======
# 실행 전: 전역 설정 + 프레임 추출 함수 + KoDF 경로 구조 셀들이 먼저 실행되어 있어야 함.
# 목적: 일부 비디오에서 Pillow fallback 저장 정상 여부 사전 검증.

import random

def quick_debug_sampling(n_real=3, n_fake=3, use_ascii_temp=False):
    """일부 KoDF 비디오만 테스트 저장하여 실제 jpg 생성 여부 확인.
    use_ascii_temp=True면 한글 경로 영향 배제 위해 C:/temp_deepfake_debug 사용."""
    print("\n[Quick Debug Sampling] 일부 비디오 테스트")
    # 기본 KoDF real/fake 경로
    video_root_train_fake = BASE_DIR / "딥페이크 변조 영상" / "1.Training" / "[원천][변조]train_dfl1_data" / "dfl1"
    video_root_train_real = BASE_DIR / "딥페이크 변조 영상" / "1.Training" / "[원천][원본]원본1" / "원본1"

    def collect(root):
        out = []
        if root.exists():
            for uuid_folder in root.glob("*"):
                if uuid_folder.is_dir():
                    for vf in uuid_folder.glob("*.mp4"):
                        out.append((uuid_folder.name, vf))
        return out

    real_all = collect(video_root_train_real)
    fake_all = collect(video_root_train_fake)
    print(f"Real 총 {len(real_all)}개, Fake 총 {len(fake_all)}개")
    if len(real_all) == 0 or len(fake_all) == 0:
        print("[WARN] 비디오 수집 실패. 경로/인코딩 점검 필요.")
        return

    real_sel = random.sample(real_all, min(n_real, len(real_all)))
    fake_sel = random.sample(fake_all, min(n_fake, len(fake_all)))

    print("선택된 Real:")
    for _, p in real_sel:
        print("  ", p)
    print("선택된 Fake:")
    for _, p in fake_sel:
        print("  ", p)

    if use_ascii_temp:
        base_out_real = Path("C:/temp_deepfake_debug/real")
        base_out_fake = Path("C:/temp_deepfake_debug/fake")
        base_out_real.mkdir(parents=True, exist_ok=True)
        base_out_fake.mkdir(parents=True, exist_ok=True)
    else:
        base_out_real = FRAMES_DIR / "_debug_real"
        base_out_fake = FRAMES_DIR / "_debug_fake"
        base_out_real.mkdir(parents=True, exist_ok=True)
        base_out_fake.mkdir(parents=True, exist_ok=True)

    # 저장 테스트 (Pillow fallback 포함)
    for uuid, vpath in real_sel:
        extract_frames_from_video(vpath, base_out_real, f"dbg_real_{uuid}_{vpath.stem}", num_samples=5, debug=True)
    for uuid, vpath in fake_sel:
        extract_frames_from_video(vpath, base_out_fake, f"dbg_fake_{uuid}_{vpath.stem}", num_samples=5, debug=True)

    # 결과 확인
    real_files = list(base_out_real.glob("*.jpg"))
    fake_files = list(base_out_fake.glob("*.jpg"))
    print(f"\n[결과] debug real 저장된 프레임: {len(real_files)}")
    print(f"[결과] debug fake 저장된 프레임: {len(fake_files)}")
    if real_files:
        print("  첫 real 파일:", real_files[0])
    if fake_files:
        print("  첫 fake 파일:", fake_files[0])

    if len(real_files) == 0 and len(fake_files) == 0:
        print("⚠️ 전혀 저장되지 않음 → 가능한 원인: 코덱/경로 인코딩/JPEG write 실패.")
        print("   해결 가이드:")
        print("   1) debug_env_opencv() 실행")
        print("   2) use_ascii_temp=True 옵션으로 재시도")
        print("   3) Pillow fallback 이미 활성 → OpenCV 빌드(JPEG/FFmpeg) 점검")
    else:
        print("✅ Quick debug 성공. 전체 샘플링 진행 가능.")

print("✅ quick_debug_sampling 정리 완료 (호출 예: quick_debug_sampling())")

✅ quick_debug_sampling 정리 완료 (호출 예: quick_debug_sampling())


In [110]:
quick_debug_sampling

<function __main__.quick_debug_sampling(n_real=3, n_fake=3, use_ascii_temp=False)>

In [111]:
run_all_sampling()
verify_frames()


[예상 프레임/용량 계획]
Train: 40,000  (KoDF 24,000 + FFPP 16,000)
Val:   10,000  (KoDF 8,000 + FFPP 2,000)
Test:  11,000  (FFPP 2,000 + CelebDF 9,000)
Total Frames: 61,000
낙관 용량 ≈ 1.70 GB  (avg 29KB)
보수 용량 ≈ 8.52 GB  (avg 146KB)
예산 20GB 대비 낙관 여유: 18.30 GB / 보수 여유: 11.48 GB

전체 데이터셋 샘플링 시작
작업 시작 시간: 2025-11-23 18:17:13.652718

KoDF 샘플링 시작 (가이드라인 적용)
발견된 Train Real 비디오: 3020
발견된 Train Fake 비디오: 5240
발견된 Val   Real 비디오: 3025
발견된 Val   Fake 비디오: 3153

[선택 결과]
Train Real: 600 / 목표 600
Train Fake: 600 / 목표 600
Val   Real: 200 / 목표 200
Val   Fake: 200 / 목표 200

[KoDF 예상 프레임] 32,000 프레임
[KoDF 예상 용량 낙관] ≈ 0.89 GB @ 29KB
[KoDF 예상 용량 보수] ≈ 4.47 GB @ 146KB

[1/4] Train Real 프레임 추출...
발견된 Train Real 비디오: 3020
발견된 Train Fake 비디오: 5240
발견된 Val   Real 비디오: 3025
발견된 Val   Fake 비디오: 3153

[선택 결과]
Train Real: 600 / 목표 600
Train Fake: 600 / 목표 600
Val   Real: 200 / 목표 200
Val   Fake: 200 / 목표 200

[KoDF 예상 프레임] 32,000 프레임
[KoDF 예상 용량 낙관] ≈ 0.89 GB @ 29KB
[KoDF 예상 용량 보수] ≈ 4.47 GB @ 146KB

[1/4] Train Real 프레임 추출

KoDF train real: 100%|██████████| 600/600 [18:05<00:00,  1.81s/it]



[2/4] Train Fake 프레임 추출...


KoDF train fake: 100%|██████████| 600/600 [17:24<00:00,  1.74s/it]



[3/4] Val Real 프레임 추출...


...

CelebDF test fake: 100%|██████████| 340/340 [00:20<00:00, 16.42it/s]




✅ Celeb-DF 샘플링 완료!

✅ 전체 샘플링 완료!
작업 종료 시간: 2025-11-23 19:23:00.387949
총 소요 시간: 65.78분

[실제 샘플링 결과 통계]
KoDF/train/real: 11998 frames
KoDF/train/fake: 12000 frames
KoDF/val/real: 4000 frames
KoDF/val/fake: 4000 frames
FFPP/train/real: 8000 frames
FFPP/train/fake: 7549 frames
FFPP/val/real: 1000 frames
FFPP/val/fake: 980 frames
FFPP/test/real: 1000 frames
FFPP/test/fake: 1000 frames
CelebDF/test/real: 1780 frames
CelebDF/test/fake: 3400 frames

총 프레임 수(실제): 56,707

메타데이터 저장 중...

샘플링 메타데이터 저장 시작
FFPP/train/fake: 7549 frames
FFPP/val/real: 1000 frames
FFPP/val/fake: 980 frames
FFPP/test/real: 1000 frames
FFPP/test/fake: 1000 frames
CelebDF/test/real: 1780 frames
CelebDF/test/fake: 3400 frames

총 프레임 수(실제): 56,707

메타데이터 저장 중...

샘플링 메타데이터 저장 시작
[DEBUG] 총 스캔된 jpg 파일 수: 56707

총 프레임 수: 56,707

[데이터셋별 통계]
label           fake   real
dataset split              
CelebDF test    3400   1780
FFPP    test    1000   1000
        train   7549   8000
        val      980   1000
KoDF    train  12000 

Verifying frames: 100%|██████████| 56707/56707 [00:04<00:00, 13631.29it/s]


✅ 모든 프레임 파일 확인 완료! (56,707개)


True

In [ ]:
# ====== 추가 진단 셀: OpenCV 빌드정보 / JPEG/PNG 저장 테스트 / ASCII 경로 테스트 ======
import textwrap

def debug_env_opencv():
    print("\n[디버그] OpenCV 빌드 정보 핵심 추출")
    build_info = cv2.getBuildInformation()
    # 핵심 키워드 검색
    keywords = ["FFMPEG", "JPEG", "libjpeg", "IPP", "PNG", "TIFF", "WebP"]
    for kw in keywords:
        found = kw in build_info
        print(f"  {kw}: {'YES' if found else 'NO'}")
    # JPEG 관련 줄 일부 표시
    print("\n[일부 빌드 문자열 내 JPEG 관련 라인]")
    snippet_lines = [l for l in build_info.splitlines() if 'jpeg' in l.lower()][:8]
    for l in snippet_lines:
        print("  ", l)

    print("\n[단순 난수 이미지 저장 테스트]")
    test_dir = Path("C:/temp_opencv_test_ascii")
    test_dir.mkdir(parents=True, exist_ok=True)
    arr = np.random.randint(0,255,(50,50,3),dtype=np.uint8)
    jpg_path = test_dir / "test.jpg"
    png_path = test_dir / "test.png"
    ok_jpg = cv2.imwrite(str(jpg_path), arr)
    ok_png = cv2.imwrite(str(png_path), arr)
    print(f"  JPG 저장 성공 여부(OpenCV): {ok_jpg} -> {jpg_path}")
    print(f"  PNG 저장 성공 여부(OpenCV): {ok_png} -> {png_path}")
    if not ok_jpg:
        # Pillow fallback 테스트
        try:
            Image.fromarray(cv2.cvtColor(arr, cv2.COLOR_BGR2RGB)).save(jpg_path, format='JPEG', quality=80)
            print("  Pillow로 JPG 저장 성공")
        except Exception as e:
            print("  Pillow JPG 저장 실패:", e)

    # 한글 경로 테스트
    korean_dir = Path("C:/temp_한글경로_테스트")
    korean_dir.mkdir(parents=True, exist_ok=True)
    k_jpg = korean_dir / "테스트이미지.jpg"
    ok_k = cv2.imwrite(str(k_jpg), arr)
    print(f"  한글 경로 JPG 저장(OpenCV) 성공 여부: {ok_k} -> {k_jpg}")
    if not ok_k:
        try:
            Image.fromarray(cv2.cvtColor(arr, cv2.COLOR_BGR2RGB)).save(k_jpg, format='JPEG', quality=80)
            print("  Pillow로 한글 경로 JPG 저장 성공")
        except Exception as e:
            print("  Pillow 한글 경로 저장 실패:", e)

    print("\n[ASCII 경로로 실제 비디오 프레임 추출 Quick Test]")
    # KoDF real 비디오 하나 선택
    video_root_train_real = BASE_DIR / "딥페이크 변조 영상" / "1.Training" / "[원천][원본]원본1" / "원본1"
    candidate = None
    if video_root_train_real.exists():
        for uuid_folder in video_root_train_real.glob('*'):
            if uuid_folder.is_dir():
                mp4s = list(uuid_folder.glob('*.mp4'))
                if mp4s:
                    candidate = mp4s[0]
                    break
    if candidate is None:
        print("  Real 비디오를 찾지 못함")
        return
    print("  선택된 비디오:", candidate)
    ascii_out = Path("C:/temp_opencv_test_ascii/frames_test")
    ascii_out.mkdir(parents=True, exist_ok=True)
    saved = extract_frames_from_video(candidate, ascii_out, "ascii_test_video", num_samples=5, debug=True)
    print(f"  저장된 프레임 수: {saved}")
    print(f"  출력 경로 내 JPG 개수: {len(list(ascii_out.glob('*.jpg')))}")
    print("\n✅ 환경 디버그 완료")

print("✅ 환경 디버그 함수 정의 완료: debug_env_opencv() 실행 가능")

✅ 환경 디버그 함수 정의 완료: debug_env_opencv() 실행 가능


### ✅ 정리된 실행 순서 가이드

1. GPU 및 경로 셀 실행 (셀 1~2)
2. 전역 설정 및 프레임 추출 함수 정의 (셀 3~4)
3. 데이터셋 샘플링 함수 정의: KoDF, FFPP, CelebDF (셀 5~7)
4. 메인 실행/메타데이터 저장/헬퍼 함수 (셀 8~9)
5. (선택) Quick Debug: `quick_debug_sampling()` (셀 10)
6. (선택) 환경 디버그: `debug_env_opencv()` (셀 11)
7. (선택) 실측 평균 보정: `calibrate_actual_frame_size(); print_expected_plan()`
8. 전체 샘플링: `run_all_sampling()`
9. (권장) 프레임 검증: `verify_frames()`
10. 메타데이터 활용: `load_sampling_metadata(split='train')` 등

#### 최소 실행 흐름 (빠른 진행)
```python
quick_debug_sampling()              # 저장 정상여부 확인
run_all_sampling()                  # 전체 샘플링
save_sampling_metadata()            # (메인에서 이미 호출되면 생략 가능)
verify_frames()                     # 누락 확인
```

#### 용량 추정 정밀화 흐름
```python
quick_debug_sampling()
calibrate_actual_frame_size()
print_expected_plan()    # 실측 평균 기반 GB 반영
run_all_sampling()
```

#### 에러 발생 시 체크
- JPEG 저장 실패 반복 → Pillow fallback 이미 활성. OpenCV 빌드: `debug_env_opencv()`.
- 한글 경로 이슈 의심 → `quick_debug_sampling(use_ascii_temp=True)`.
- 메타데이터 비어있음 → 프레임 디렉토리 삭제 후 재실행 또는 경로 재확인.

이제 중복/불필요 셀 정리 완료. 필요 시 추가 개선(프레임 수 조정, 품질 변경) 요청하세요.

## 🆕 KoDF Identity 기반 샘플링 (신규 데이터셋)

아래 셀들은 새로 다운로드한 KoDF 데이터를 identity(인물) 기준으로 샘플링합니다:
- **목표**: 100~150명의 identity 확보
- **프레임 샘플링**: 영상당 16~32장
- **출력**: 기존 `Deep_Fake_datasets` 폴더 구조에 통합

### 📋 실행 가이드

#### 1단계: 경로 설정
```python
# 다운로드한 KoDF 데이터 경로로 수정
NEW_KODF_ROOT = BASE_DIR / "다운로드폴더명"
```

#### 2단계: 폴더 구조 확인
```python
explore_new_kodf_structure(NEW_KODF_ROOT)
```

#### 3단계: `collect_videos_by_identity()` 수정
폴더 구조에 따라 함수 내부 로직 수정:
- **패턴 A**: `identity_folder/video.mp4` → 기본 코드 그대로 사용
- **패턴 B**: `video_id123_person456.mp4` → 파일명 파싱 로직 활성화

#### 4단계: 샘플링 실행
```python
run_kodf_identity_sampling()
```

#### 5단계: 결과 확인
```python
# Deep_Fake_datasets/KoDF/ 폴더 내 프레임 확인
for split in ['train', 'val', 'test']:
    for label in ['real', 'fake']:
        path = OUTPUT_DIR / split / label
        if path.exists():
            print(f"{split}/{label}: {len(list(path.glob('*.jpg')))} frames")
```

#### 💡 주요 파라미터 조정
- `MIN_IDENTITIES`, `MAX_IDENTITIES`: identity 수 범위
- `FRAMES_PER_VIDEO_MIN`, `FRAMES_PER_VIDEO_MAX`: 영상당 프레임 수
- `IDENTITY_SPLIT_RATIO`: train/val/test 비율